# Part 4 — Best Model Extended Training & SVG Generation

**Model:** SP Large (33M params, d_model=512, 10 layers, lr=3e-3)

**Tasks:**
- 4.1: Train Large for 3 epochs (~15 min on A100)
- 4.2/4.3: Generate 10 unconditional + 5 prefix-conditioned SVG samples
- 4.4: Compute perplexity, XML validity rate, SVG render rate

**Why Large and not XL?** The LR sweep found `3e-3` as optimal. Large was trained
with `3e-3` in Part 2, achieving val_loss=1.1543 — the best result across all 5 sizes.
XL was accidentally trained with `1e-3` (sub-optimal) and scored 1.1847. Large is the
legitimate winner of the Part 2 experimental protocol.

In [ ]:
# ── 1. Mount Google Drive ─────────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')
print('Drive mounted ✓')

In [ ]:
# ── 2. GPU check ──────────────────────────────────────────────────────────────
import torch
device = 'cuda' if torch.cuda.is_available() else 'cpu'
if device == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'Memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB')
else:
    print('No GPU — running on CPU (will be slow)')
print('PyTorch:', torch.__version__)

In [ ]:
# ── 3. Install dependencies + clone repo ──────────────────────────────────────
import subprocess, sys, os

subprocess.run([sys.executable, '-m', 'pip', 'install',
                'tokenizers', 'cairosvg', 'lxml', '-q'], check=True)
print('Dependencies installed ✓')

REPO_URL = 'https://github.com/shubham739/ML-Final-Project.git'
REPO_DIR = '/content/ML-Final-Project'

if os.path.exists(f'{REPO_DIR}/.git'):
    print('Repo exists — pulling latest...')
    subprocess.run(['git', '-C', REPO_DIR, 'pull'], check=True)
else:
    subprocess.run(['git', 'clone', REPO_URL, REPO_DIR], check=True)
    print('Repo cloned ✓')

os.chdir(REPO_DIR)
print('Working directory:', os.getcwd())

In [ ]:
# ── 4. Configure paths ────────────────────────────────────────────────────────
# ⚠️  Edit DRIVE_ROOT only if your project lives in a different Drive folder.
DRIVE_ROOT = '/content/drive/MyDrive/ML-Final-Project'

DATA_DIR   = f'{DRIVE_ROOT}/data/tokenized'
OUTPUT_DIR = f'{DRIVE_ROOT}/outputs'
CONFIG_DIR = f'{REPO_DIR}/configs'

for d in [f'{OUTPUT_DIR}/checkpoints', f'{OUTPUT_DIR}/logs',
           f'{OUTPUT_DIR}/results',     f'{OUTPUT_DIR}/plots',
           f'{OUTPUT_DIR}/generated',   f'{OUTPUT_DIR}/generated_png']:
    os.makedirs(d, exist_ok=True)

for fname in ['train.npy', 'val.npy', 'test.npy']:
    p = f'{DATA_DIR}/{fname}'
    assert os.path.exists(p), f'Missing: {p} — upload data to Drive first'

print(f'REPO_DIR:   {REPO_DIR}')
print(f'DATA_DIR:   {DATA_DIR}')
print(f'OUTPUT_DIR: {OUTPUT_DIR}')
print('All data files found ✓')

In [ ]:
# ── 5. Imports (self-contained — safe to re-run after reset) ─────────────────
# Re-asserts REPO_DIR on sys.path so imports work even if cell 3 was skipped.

import json, math, time
import numpy as np
import torch
import torch.nn as nn

if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

from scripts.model import GPT, ModelConfig
from scripts.train import load_data, get_batch, compute_val_loss, get_lr

print('Imports OK ✓')

## Task 4.1 — Extended Training: Large Model, 3 Epochs

In [ ]:
# ── 6. Training setup ─────────────────────────────────────────────────────────
CONFIG_PATH  = f'{CONFIG_DIR}/large.json'
LR           = 3e-3
N_EPOCHS     = 3
BATCH_SIZE   = 4
EFF_TOKENS   = 65536
WEIGHT_DECAY = 0.1
WARMUP_FRAC  = 0.05
GRAD_CLIP    = 1.0
EVAL_EVERY   = 100
SEED         = 42
USE_BF16     = (device == 'cuda')

torch.manual_seed(SEED)
np.random.seed(SEED)

with open(CONFIG_PATH) as f:
    cfg_dict = json.load(f)
config = ModelConfig.from_dict(cfg_dict)
model  = GPT(config).to(device)
n_params = model.count_parameters()
print(f'Model: {cfg_dict["name"]}  ({n_params/1e6:.2f}M params)')
print(f'Config: d_model={config.d_model}  n_layers={config.n_layers}  n_heads={config.n_heads}')

train_data = load_data(f'{DATA_DIR}/train.npy')
val_data   = load_data(f'{DATA_DIR}/val.npy')
print(f'Train tokens: {len(train_data):,}  |  Val tokens: {len(val_data):,}')

block_size      = config.block_size
grad_accum      = max(1, EFF_TOKENS // (BATCH_SIZE * block_size))
tokens_per_step = BATCH_SIZE * block_size * grad_accum
steps_per_epoch = math.ceil(len(train_data) / tokens_per_step)
total_steps     = steps_per_epoch * N_EPOCHS
warmup_steps    = max(1, int(total_steps * WARMUP_FRAC))
min_lr          = LR * 0.1

print(f'Steps/epoch: {steps_per_epoch}  |  Total steps: {total_steps}  |  Warmup: {warmup_steps}')
print(f'Est. time @ 5 min/epoch: ~{N_EPOCHS * 5} min')

optimizer = torch.optim.AdamW(
    model.parameters(), lr=LR,
    betas=(0.9, 0.95), eps=1e-8, weight_decay=WEIGHT_DECAY
)
amp_ctx = (torch.autocast(device, dtype=torch.bfloat16)
           if USE_BF16 else torch.autocast(device, enabled=False))

ckpt_dir = f'{OUTPUT_DIR}/checkpoints/{cfg_dict["name"]}_extended'
os.makedirs(ckpt_dir, exist_ok=True)

In [ ]:
# ── 7. Training loop ──────────────────────────────────────────────────────────
train_log   = []
t_start     = time.time()
tokens_seen = 0
peak_mem    = 0.0
model.train()

for step in range(total_steps):
    epoch = step // steps_per_epoch + 1

    current_lr = get_lr(step, warmup_steps, total_steps, LR, min_lr)
    for pg in optimizer.param_groups:
        pg['lr'] = current_lr

    optimizer.zero_grad(set_to_none=True)
    step_loss = 0.0
    for _ in range(grad_accum):
        x, y = get_batch(train_data, block_size, BATCH_SIZE, torch.device(device))
        with amp_ctx:
            _, loss = model(x, y)
        (loss / grad_accum).backward()
        step_loss += loss.item() / grad_accum

    nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
    optimizer.step()
    tokens_seen += tokens_per_step

    if device == 'cuda':
        mem = torch.cuda.max_memory_allocated() / 1024**2
        peak_mem = max(peak_mem, mem)

    if step % EVAL_EVERY == 0 or step == total_steps - 1:
        elapsed = time.time() - t_start
        tps = tokens_seen / elapsed if elapsed > 0 else 0
        val_loss = compute_val_loss(model, val_data, block_size,
                                    torch.device(device), max_batches=50)
        train_log.append({'step': step, 'epoch': epoch,
                          'train_loss': round(step_loss, 5),
                          'val_loss': round(val_loss, 5),
                          'lr': round(current_lr, 8)})
        pct = 100 * step / total_steps
        print(f'  ep{epoch} step {step:5d}/{total_steps} ({pct:4.1f}%)  '
              f'train={step_loss:.4f}  val={val_loss:.4f}  '
              f'lr={current_lr:.2e}  {tps:,.0f} tok/s  mem={peak_mem:.0f}MB')

    # Save checkpoint at end of each epoch
    if (step + 1) % steps_per_epoch == 0:
        ep = (step + 1) // steps_per_epoch
        val_loss_full = compute_val_loss(model, val_data, block_size,
                                         torch.device(device))
        ckpt_path = f'{ckpt_dir}/checkpoint_epoch{ep}.pt'
        torch.save({
            'step': step + 1, 'epoch': ep,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'config': cfg_dict,
            'val_loss': val_loss_full, 'lr': LR,
        }, ckpt_path)
        print(f'\n>>> Epoch {ep} — val_loss={val_loss_full:.4f} — saved {ckpt_path}\n')

total_time = time.time() - t_start
print(f'\nTraining complete: {total_time/60:.1f} min  |  Peak mem: {peak_mem:.0f} MB')

In [ ]:
# ── 8. Save final checkpoint + training log ───────────────────────────────────
final_val = compute_val_loss(model, val_data, block_size, torch.device(device))
CKPT_PATH = f'{ckpt_dir}/checkpoint_final.pt'
torch.save({
    'step': total_steps, 'epoch': N_EPOCHS,
    'model_state_dict': model.state_dict(),
    'optimizer_state_dict': optimizer.state_dict(),
    'config': cfg_dict,
    'val_loss': final_val, 'lr': LR,
}, CKPT_PATH)

log = {
    'name': cfg_dict['name'] + '_extended', 'n_epochs': N_EPOCHS,
    'total_steps': total_steps, 'lr': LR, 'val_loss': final_val,
    'total_time_seconds': round(total_time, 1),
    'tokens_per_second': round(tokens_seen / total_time, 1),
    'peak_memory_mb': round(peak_mem, 1),
    'train_losses': train_log,
}
with open(f'{OUTPUT_DIR}/logs/large_extended_lr3e-3.json', 'w') as f:
    json.dump(log, f, indent=2)

print(f'Final val_loss: {final_val:.4f}')
print(f'Checkpoint:     {CKPT_PATH}')

In [ ]:
# ── 9. Training curve plot ────────────────────────────────────────────────────
import matplotlib.pyplot as plt

steps  = [e['step']       for e in train_log]
t_loss = [e['train_loss'] for e in train_log]
v_loss = [e['val_loss']   for e in train_log]

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(steps, t_loss, label='Train loss', alpha=0.7, linewidth=1.5)
ax.plot(steps, v_loss, label='Val loss', linewidth=2, color='tomato')
for ep in range(1, N_EPOCHS):
    ax.axvline(ep * steps_per_epoch, color='gray', linestyle=':', alpha=0.6)
ax.set_xlabel('Training Step', fontsize=12)
ax.set_ylabel('Loss', fontsize=12)
ax.set_title(f'Task 4.1 — Large Model Extended Training ({N_EPOCHS} epochs, lr=3e-3)', fontsize=12)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
fig.tight_layout()
fig.savefig(f'{OUTPUT_DIR}/plots/large_extended_training_curve.png', dpi=150)
plt.show()
print('Saved: outputs/plots/large_extended_training_curve.png')

## Tasks 4.2 & 4.3 — SVG Generation

10 unconditional samples across 3 temperatures (T=0.5, 0.8, 1.0) + 5 prefix-conditioned samples.

In [ ]:
# ── 10a. Load checkpoint + generator setup ───────────────────────────────────
# Run this before any generation cell. Safe to run even if training was skipped.

from tokenizers import Tokenizer

# ── Load checkpoint ──────────────────────────────────────────────────────────
CKPT_PATH = f'{OUTPUT_DIR}/checkpoints/large_extended/checkpoint_final.pt'

assert os.path.exists(CKPT_PATH), (
    f'Checkpoint not found: {CKPT_PATH}\n'
    'Run the training cells first (cells 6-9), or check your Drive path.'
)

ckpt       = torch.load(CKPT_PATH, map_location=device, weights_only=False)
cfg_dict   = ckpt['config']
config     = ModelConfig.from_dict(cfg_dict)
model      = GPT(config).to(device)
model.load_state_dict(ckpt['model_state_dict'])
model.eval()

n_params   = model.count_parameters()
final_val  = ckpt['val_loss']
N_EPOCHS   = ckpt.get('epoch', 3)
block_size = config.block_size

print(f'Loaded: {cfg_dict["name"]}  ({n_params/1e6:.2f}M params)')
print(f'Epochs: {N_EPOCHS}  —  val_loss = {final_val:.4f}')

# ── Tokenizer + generate_svg ─────────────────────────────────────────────────
tokenizer    = Tokenizer.from_file(f'{REPO_DIR}/tokenizer/tokenizer.json')
EOS_ID       = 2
UNCOND_PROMPT = '<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 24 24">'

def ids_to_svg(ids):
    clean = [i for i in ids if i not in (EOS_ID, 0)]
    return tokenizer.decode(clean)

@torch.no_grad()
def generate_svg(prefix=None, max_new_tokens=400,
                 temperature=0.8, top_k=50, top_p=None):
    model.eval()
    prompt_text = prefix if prefix else UNCOND_PROMPT
    prompt_ids  = tokenizer.encode(prompt_text).ids
    idx = torch.tensor([prompt_ids], dtype=torch.long, device=device)
    out = model.generate(idx, max_new_tokens=max_new_tokens,
                         temperature=temperature, top_k=top_k,
                         top_p=top_p, eos_id=EOS_ID)
    all_ids = out[0].tolist()
    new_ids = all_ids[len(prompt_ids):]
    return prompt_text + ids_to_svg(new_ids)

gen_dir = f'{OUTPUT_DIR}/generated'
print('Generator ready ✓')
print(f'Unconditional prompt: {UNCOND_PROMPT}')
print(f'Max new tokens per sample: 400')

In [ ]:
# ── 11. Unconditional generation (10 samples, 3 temperatures) ─────────────────
torch.manual_seed(42)

# (temperature, top_k, n_samples)
temp_configs = [(0.5, 50, 4), (0.8, 50, 4), (1.0, 50, 2)]

all_unconditional = []
idx = 1
for temp, tk, n in temp_configs:
    print(f'\nTemperature={temp}, top_k={tk}, n={n}')
    for _ in range(n):
        t0 = time.time()
        svg = generate_svg(temperature=temp, top_k=tk)
        fname = f'unconditional_{idx:02d}_T{temp}.svg'
        with open(f'{gen_dir}/{fname}', 'w') as f:
            f.write(svg)
        print(f'  [{idx:02d}] {len(svg)} chars  ({time.time()-t0:.1f}s)  {fname}')
        all_unconditional.append({'filename': fname, 'temperature': temp,
                                   'top_k': tk, 'svg': svg})
        idx += 1

print(f'\nGenerated {len(all_unconditional)} unconditional samples ✓')

In [ ]:
# ── 12. Prefix-conditioned generation (5 samples) ─────────────────────────────
PREFIXES = [
    ('<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 24 24">'
     '<circle cx="12" cy="12" r="10" fill="none" stroke="black" stroke-width="2"/>',
     'circle_face'),
    ('<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 24 24">'
     '<path d="M4 4 L20 4 L20 20"',
     'open_path'),
    ('<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 24 24">'
     '<g><rect x="2" y="2" width="8" height="8" fill="steelblue"/>',
     'group_rect'),
    ('<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 24 24">'
     '<line x1="2" y1="12" x2="18" y2="12" stroke="black" stroke-width="2"/>',
     'arrow_shaft'),
    ('<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 24 24">'
     '<path d="M12 2 L12 22 M12 2 L4 8"',
     'half_icon'),
]

torch.manual_seed(42)
all_prefix = []

for i, (prefix, tag) in enumerate(PREFIXES, 1):
    t0 = time.time()
    svg = generate_svg(prefix=prefix, temperature=0.8, top_k=50)
    fname = f'prefix_{i:02d}_{tag}.svg'
    with open(f'{gen_dir}/{fname}', 'w') as f:
        f.write(svg)
    print(f'[{i}] {tag}: {len(prefix)}→{len(svg)} chars  ({time.time()-t0:.1f}s)')
    all_prefix.append({'filename': fname, 'tag': tag, 'prefix': prefix, 'svg': svg})

print(f'\nGenerated {len(all_prefix)} prefix-conditioned samples ✓')

## Task 4.4 — Quantitative Evaluation

In [ ]:
# ── 13. Test set perplexity ───────────────────────────────────────────────────
import math

test_data  = load_data(f'{DATA_DIR}/test.npy')
avg_loss   = compute_val_loss(model, test_data, block_size, torch.device(device))
perplexity = math.exp(avg_loss)
print(f'Test set avg cross-entropy: {avg_loss:.4f}')
print(f'Test set perplexity:        {perplexity:.4f}')

In [ ]:
# ── 14. XML validity, render rate, structural validity ────────────────────────
from lxml import etree
try:
    import cairosvg
    HAS_CAIRO = True
except ImportError:
    print('[WARN] cairosvg not available — render rate will be 0')
    HAS_CAIRO = False

png_dir = f'{OUTPUT_DIR}/generated_png'
all_samples = all_unconditional + all_prefix
results = []

for s in all_samples:
    svg = s['svg']

    try:
        etree.fromstring(svg.encode())
        xml_valid = True
    except Exception:
        xml_valid = False

    renderable = False
    if HAS_CAIRO and xml_valid:
        try:
            png = cairosvg.svg2png(bytestring=svg.encode())
            png_path = f"{png_dir}/{s['filename'].replace('.svg', '.png')}"
            with open(png_path, 'wb') as f:
                f.write(png)
            renderable = True
        except Exception:
            pass

    text = svg.strip()
    has_svg_root = text.startswith('<svg') and '</svg>' in text
    has_shape    = any(f'<{t}' in text for t in
                       ['path', 'circle', 'rect', 'line', 'polygon', 'ellipse', 'g'])
    structural   = has_svg_root and text.endswith('</svg>') and has_shape

    results.append({
        'filename': s['filename'],
        'type': 'prefix' if 'prefix' in s['filename'] else 'unconditional',
        'xml_valid': xml_valid, 'renderable': renderable,
        'structural': structural, 'length': len(svg),
    })

n = len(results)
xml_rate    = sum(r['xml_valid']  for r in results) / n
render_rate = sum(r['renderable'] for r in results) / n
struct_rate = sum(r['structural'] for r in results) / n

print(f'Metrics over {n} samples:')
print(f'  XML validity rate:   {xml_rate:.1%}  ({sum(r["xml_valid"] for r in results)}/{n})')
print(f'  SVG render rate:     {render_rate:.1%}  ({sum(r["renderable"] for r in results)}/{n})')
print(f'  Structural validity: {struct_rate:.1%}  ({sum(r["structural"] for r in results)}/{n})')
print(f'  Avg length:          {sum(r["length"] for r in results)//n} chars')

In [ ]:
# ── 15. Save all results ──────────────────────────────────────────────────────
eval_out = {
    'model_name': cfg_dict['name'] + '_extended',
    'n_epochs': N_EPOCHS, 'final_val_loss': final_val,
    'test_perplexity': perplexity,
    'xml_validity_rate': xml_rate, 'render_rate': render_rate,
    'structural_rate': struct_rate,
    'n_unconditional': len(all_unconditional),
    'n_prefix_conditioned': len(all_prefix),
    'per_sample': results,
}
with open(f'{OUTPUT_DIR}/results/evaluation_results.json', 'w') as f:
    json.dump(eval_out, f, indent=2)
print('Saved: evaluation_results.json ✓')

gen_meta = {
    'checkpoint': CKPT_PATH,
    'samples': [{'filename': s['filename'],
                 'temperature': s.get('temperature', 0.8),
                 'type': 'prefix' if 'prefix' in s['filename'] else 'unconditional'}
                for s in all_samples]
}
with open(f'{OUTPUT_DIR}/results/generation_results.json', 'w') as f:
    json.dump(gen_meta, f, indent=2)
print('Saved: generation_results.json ✓')

## Task 4.5 — Qualitative Analysis

In [ ]:
# ── 16. Display generated samples ─────────────────────────────────────────────
from IPython.display import Image, display

print('=== UNCONDITIONAL SAMPLES (first 4) ===')
for s in all_unconditional[:4]:
    print(f"\n--- {s['filename']} (T={s['temperature']}) ---")
    print(s['svg'][:300] + ('...' if len(s['svg']) > 300 else ''))
    png_path = f"{png_dir}/{s['filename'].replace('.svg', '.png')}"
    if os.path.exists(png_path):
        display(Image(png_path, width=200))

In [ ]:
# ── 17. Prefix-conditioned display ────────────────────────────────────────────
print('=== PREFIX-CONDITIONED SAMPLES ===')
for s in all_prefix:
    print(f"\n--- {s['filename']} ---")
    print(f"PREFIX:     {s['prefix']}")
    completion = s['svg'][len(s['prefix']):len(s['prefix'])+200]
    print(f"COMPLETION: ...{completion}")
    png_path = f"{png_dir}/{s['filename'].replace('.svg', '.png')}"
    if os.path.exists(png_path):
        display(Image(png_path, width=200))

In [ ]:
# ── 18. Part 4 summary ────────────────────────────────────────────────────────
print('=' * 60)
print('PART 4 SUMMARY')
print('=' * 60)
print(f'Model:              Large ({n_params/1e6:.1f}M params), SP, lr=3e-3')
print(f'Extended training:  {N_EPOCHS} epochs')
print(f'Final val_loss:     {final_val:.4f}')
print(f'Test perplexity:    {perplexity:.4f}')
print(f'Unconditional:      {len(all_unconditional)} samples')
print(f'Prefix-conditioned: {len(all_prefix)} samples')
print(f'XML validity:       {xml_rate:.1%}')
print(f'Render rate:        {render_rate:.1%}')
print(f'Structural valid:   {struct_rate:.1%}')
print('=' * 60)